# Stage 2 — In-Sample Optimization

Per [METHODOLOGY.md](../METHODOLOGY.md) and the parameter space locked in [01_hypothesis.ipynb](01_hypothesis.ipynb):

**Grid:** 5 × 5 × 5 × 4 × 3 = **1,500 trials**.  
**Data:** IS partition only (60% earliest, ≈ 22k 1h bars).  
**Cost mode:** HORROR (the only mode used for any reported number).  

This notebook loads the search log produced by `quant.optimization.grid_search` and:
1. Verifies the recorded `N_trials` (used by Stage 5's DSR).
2. Inspects the distribution of metrics across all 1,500 candidates.
3. Selects the **top 10 by Sharpe** for OOS-1 validation in Stage 3.
4. Persists the selection to `data/processed/top10_is_candidates.json`.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

RESULTS = Path('../results')
DATA_PROC = Path('../data/processed')
SEARCH_LOG = RESULTS / 'tables' / 'is_search_log.csv'

## 1. Load the search log

If `is_search_log.csv` is missing, re-run the grid search:
```bash
uv run python -m quant.optimization.grid_search
```
(Or use the inline snippet in `02_data_quality.ipynb`.)

In [ ]:
results = pd.read_csv(SEARCH_LOG)
n_trials = int(results['n_trials'].iloc[0])
print(f'Trials loaded: {len(results):,}  (recorded N_trials = {n_trials:,})')
assert len(results) == n_trials, 'Trial count mismatch — abort'
assert n_trials == 1500, 'Recorded N_trials must equal the locked 1500'
results.head(3)

## 2. Summary statistics across the grid

The distribution shape matters — a strategy that's only positive at a single point in the grid is unlikely to survive OOS.

In [ ]:
metric_cols = ['n_trades', 'win_rate', 'pf', 'sharpe', 'max_dd', 'cagr']
summary = results[metric_cols].describe(percentiles=[0.10, 0.50, 0.90, 0.99]).round(3)
summary

In [ ]:
n_pf_above_1 = int((results['pf'] > 1.0).sum())
n_sharpe_positive = int((results['sharpe'] > 0).sum())
print(f'Trials with PF > 1.0:       {n_pf_above_1:>5,} / {len(results):,}  ({n_pf_above_1/len(results):.1%})')
print(f'Trials with Sharpe > 0:     {n_sharpe_positive:>5,} / {len(results):,}  ({n_sharpe_positive/len(results):.1%})')

## 3. Distribution shape — Sharpe + PF + DD

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, color in zip(axes, ['sharpe', 'pf', 'max_dd'], ['#3b82f6', '#10b981', '#ef4444']):
    ax.hist(results[col], bins=40, color=color, alpha=0.7, edgecolor='black', linewidth=0.3)
    ax.axvline(0 if col == 'sharpe' else 1.0 if col == 'pf' else 0.3, color='black', linestyle='--', lw=1, alpha=0.6)
    ax.set_title(f'{col.upper()} across {len(results):,} trials')
    ax.set_xlabel(col)
    ax.set_ylabel('Trials')
    ax.grid(alpha=0.3)

plt.tight_layout()
out = RESULTS / 'figures' / '02_is_metric_distributions.png'
out.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out, dpi=120, bbox_inches='tight')
print(f'Wrote: {out}')
plt.show()

## 4. Top 10 by Sharpe — candidates for OOS-1 validation

In [ ]:
param_cols = ['min_range_pct', 'extreme_proximity_pct', 'stop_buffer_pct', 'tp_fraction', 'time_stop_hours']
display_cols = param_cols + ['n_trades', 'win_rate', 'pf', 'sharpe', 'max_dd', 'cagr']
top10 = results.sort_values('sharpe', ascending=False).head(10).reset_index(drop=True)
top10[display_cols].round(4)

In [ ]:
candidates = []
for _, row in top10.iterrows():
    entry = {k: (int(row[k]) if k == 'time_stop_hours' else float(row[k])) for k in param_cols}
    entry['is_sharpe'] = float(row['sharpe'])
    entry['is_pf'] = float(row['pf'])
    entry['is_max_dd'] = float(row['max_dd'])
    entry['is_n_trades'] = int(row['n_trades'])
    candidates.append(entry)

DATA_PROC.mkdir(parents=True, exist_ok=True)
out_json = DATA_PROC / 'top10_is_candidates.json'
out_json.write_text(json.dumps(candidates, indent=2))
print(f'Wrote: {out_json}')
print()
print(json.dumps(candidates, indent=2))

## 5. Parameter sensitivity — Sharpe heatmaps

If the high-Sharpe region is an isolated spike, the strategy is fragile. A smooth ridge is healthier.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

pivot1 = results.pivot_table(values='sharpe', index='extreme_proximity_pct', columns='min_range_pct', aggfunc='mean')
sns.heatmap(pivot1, annot=True, fmt='.2f', cmap='RdYlGn', center=0, ax=axes[0], cbar_kws={'label': 'Mean Sharpe'})
axes[0].set_title('Avg Sharpe: extreme_proximity × min_range')

pivot2 = results.pivot_table(values='sharpe', index='tp_fraction', columns='stop_buffer_pct', aggfunc='mean')
sns.heatmap(pivot2, annot=True, fmt='.2f', cmap='RdYlGn', center=0, ax=axes[1], cbar_kws={'label': 'Mean Sharpe'})
axes[1].set_title('Avg Sharpe: tp_fraction × stop_buffer')

plt.tight_layout()
out = RESULTS / 'figures' / '03_is_param_heatmaps.png'
fig.savefig(out, dpi=120, bbox_inches='tight')
print(f'Wrote: {out}')
plt.show()

## Stage 2 complete

- 1,500 trials evaluated on IS only (N_trials recorded for Stage 5 DSR).
- Top 10 candidates by Sharpe persisted to `data/processed/top10_is_candidates.json`.
- Distribution + sensitivity diagnostics written to `results/figures/`.

**Important — no acceptance gate applies at Stage 2.** Even if the best Sharpe is impressive, Stage 3 (OOS-1 validation) is the first real-money-relevant test. Proceed to [04_oos_validation.ipynb](04_oos_validation.ipynb).